# Assignment 2: Milestone I Natural Language Processing
## Task 1. Basic Text Pre-processing
#### Student Name: Ngo Trong Nhan
#### Student ID: 4196976


Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
You should give a brief information of this assessment task here.

<span style="color: red"> Note that this is a sample notebook only. You will need to fill in the proper markdown and code blocks. You might also want to make necessary changes to the structure to meet your own needs. Note also that any generic comments written in this notebook are to be removed and replace with your own words.</span>

## Importing libraries 

In [88]:
# Code to import libraries as you need in this assessment, e.g.,
import os
import re
import textwrap
import pandas as pd
from collections import Counter
from dotenv import load_dotenv
load_dotenv()


True

# Helpers

In [89]:
TOKENIZER_PATTERN = re.compile(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?") # From documentation
STOPWORDS_PATH = "./data/stopwords_en.txt"
ENHANCED_PATH = "./data/stopwords_en_enhancement.txt" 
VOCAB_PATH = "./data/vocab.txt"
OUTPUT_PATH = "./data/processed.csv"

In [52]:
def load_stopwords(path: str) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        words: list[str] = [line.strip().lower() for line in f if line.strip()]
    return words


def basic_report(words: list[str]) -> dict:
    """Quick sanity checks on the raw file."""
    unique: list[str] = sorted(set(words))
    duplicates: list[str] = [w for w in unique if words.count(w) > 1]
    invalid_tokens: list[str] = [
        w for w in unique
        if not TOKENIZER_PATTERN.fullmatch(w)   # won't survive Step 2 tokenizer
    ]
    single_char: list[str] = [w for w in unique if len(w) < 2]  # will be dropped by Step 4

    return {
        "total_lines"     : len(words),
        "unique_words"    : len(unique),
        "duplicates"      : duplicates,
        "invalid_tokens"  : invalid_tokens,   # won't match the Task-1 regex
        "single_char"     : single_char,       # dropped by Step 4 anyway
        "unique_list"     : unique,
    }


def extract_section(label: str, text: str) -> str:
    pattern = rf"{label}:\s*(.*?)(?=\n[A-Z_]+:|\Z)"
    m = re.search(pattern, text, re.DOTALL)
    return m.group(1).strip() if m else ""


def parse_claude_response(text: str) -> dict:
    """Extract the three sections from Claude's structured reply."""

    audit_raw   = extract_section("AUDIT_ISSUES", text)
    suggest_raw = extract_section("SUGGESTED_ADDITIONS", text)
    summary_raw = extract_section("SUMMARY", text)

    # Parse audit issues: "<word> | <reason>"
    audit_issues = []
    for line in audit_raw.splitlines():
        line = line.strip()
        if "|" in line:
            parts = line.split("|", 1)
            audit_issues.append({
                "word"  : parts[0].strip().lower(),
                "reason": parts[1].strip(),
            })

    # Parse suggested additions (one word per line)
    suggestions = []
    for line in suggest_raw.splitlines():
        word = line.strip().lower().lstrip("•-* ")
        if word and TOKENIZER_PATTERN.fullmatch(word) and len(word) >= 2:
            suggestions.append(word)

    return {
        "audit_issues": audit_issues,
        "suggestions" : suggestions,
        "summary"     : summary_raw,
    }


def build_enhanced_list(
    original: list[str],
    parsed: dict,
    report: dict,
) -> list[str]:
    """
    Start from the cleaned original, remove words flagged by the audit,
    add Claude's suggestions, de-duplicate, sort.
    Only keep words that survive the Task-1 regex and have length >= 2.
    """
    flagged_words = {item["word"] for item in parsed["audit_issues"]}
    base = set()

    for w in report["unique_list"]:
        if len(w) < 2:
            continue                            # Step 4 will drop these anyway
        if not TOKENIZER_PATTERN.fullmatch(w):
            continue                            # won't be tokenized at all
        if w in flagged_words:
            continue                            # Claude flagged as non-stopword
        base.add(w)

    # Add Claude's suggestions
    for w in parsed["suggestions"]:
        base.add(w)

    return sorted(base)


def save_enhanced(words: list[str], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(words) + "\n")
    print(f"\n[Saved] {len(words)} stopwords → {path}")


def print_report(report: dict, parsed: dict, enhanced: list[str], original: list[str]) -> None:
    print("\n" + "=" * 60)
    print("  STOPWORDS INVESTIGATION REPORT")
    print("=" * 60)

    print(f"\n[Original file]")
    print(f"  Total lines  : {report['total_lines']}")
    print(f"  Unique words : {report['unique_words']}")

    if report["duplicates"]:
        print(f"  Duplicates   : {report['duplicates']}")
    else:
        print(f"  Duplicates   : none")

    if report["invalid_tokens"]:
        print(f"  Invalid (regex won't capture): {report['invalid_tokens']}")

    if report["single_char"]:
        print(f"  Single-char (Step 4 removes): {report['single_char']}")

    print(f"\n[Claude Audit — words to reconsider]")
    if parsed["audit_issues"]:
        for item in parsed["audit_issues"]:
            print(f"  - {item['word']:<20} {item['reason']}")
    else:
        print("  None flagged.")

    print(f"\n[Claude Suggestions — new additions]")
    if parsed["suggestions"]:
        print("  " + ", ".join(parsed["suggestions"]))
    else:
        print("  None suggested.")

    print(f"\n[Claude Summary]")
    for line in parsed["summary"].splitlines():
        print(f"  {line}")

    added   = set(enhanced) - set(original)
    removed = set(original) - set(enhanced)
    print(f"\n[Enhanced list]")
    print(f"  Words added   : {len(added)}   → {sorted(added) if added else 'none'}")
    print(f"  Words removed : {len(removed)} → {sorted(removed) if removed else 'none'}")
    print(f"  Final count   : {len(enhanced)}")
    print("=" * 60)

In [ ]:
class ClaudeTextExpert:
    """Manage stopword-list auditing via the Claude API."""
 
    _SAMPLE_REVIEWS   = 300   # reviews to sample for prompt context
    _SAMPLE_TOP_TOKENS = 60   # top-N tokens to surface in the prompt
 
    # ── Construction ──────────────────────────────────────────────────────────
 
    def __init__(self, model: str = "claude-sonnet-4-6", max_tokens: int = 2048):
        import anthropic
        self.model      = model
        self.max_tokens = max_tokens
        self.client     = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env
 
    # ── Public helpers ────────────────────────────────────────────────────────
 
    @staticmethod
    def show_api_key() -> None:
        key = os.getenv("ANTHROPIC_API_KEY", "")
        if not key:
            print("[API] ANTHROPIC_API_KEY not set.")
            return
        print(f"[API] Key starts with: {key[:8]}…")
 
    # ── Sampling (classmethod — no API needed) ────────────────────────────────
 
    @classmethod
    def get_sample_reviews(
        cls,
        reviews: "pd.DataFrame | list[str]",
        n_reviews: int  = _SAMPLE_REVIEWS,
        top_n: int      = _SAMPLE_TOP_TOKENS,
        seed: int       = 42,
    ) -> dict | None:
        """
        Build the context payload that will be injected into the Claude prompt.
 
        Parameters
        ----------
        reviews : pd.DataFrame or list[str]
            Post-Step-4 data.  Either:
              • DataFrame with a 'review_text' column of space-separated token strings, OR
              • list[str] where each element is one post-Step-4 token stream.
            The caller is responsible for having applied Steps 1-4
            (tokenize → lowercase → remove len<2) before passing data in.
        n_reviews : int   — rows to sample (default 300)
        top_n     : int   — most-frequent tokens to report (default 60)
        seed      : int   — random seed for reproducibility
        """
        # Normalise to flat list[str]
        if isinstance(reviews, pd.DataFrame):
            if "review_text" not in reviews.columns:
                raise ValueError("DataFrame must contain a 'review_text' column.")
            streams_all: list[str] = reviews["review_text"].dropna().astype(str).tolist()
        elif isinstance(reviews, list):
            streams_all = [str(r) for r in reviews if r]
        else:
            raise TypeError(f"reviews must be a DataFrame or list[str], got {type(reviews)}")
 
        total = len(streams_all)
        if total == 0:
            print("[Sampler] No reviews provided — skipping.")
            return None
 
        import random
        random.seed(seed)
        k = min(n_reviews, total)
        sampled = random.sample(streams_all, k)
 
        corpus_counter: Counter = Counter()
        for stream in sampled:
            corpus_counter.update(stream.split())
 
        print(f"[Sampler] {total:,} reviews received — sampled {k} for Claude context.")
        return {
            "sampled_streams": sampled,
            "top_tokens"     : corpus_counter.most_common(top_n),
            "total_reviews"  : total,
            "n_sampled"      : k,
        }
 
    # ── Audit entry-point ─────────────────────────────────────────────────────
 
    def claude_audit(self, report: dict, samples: dict | None = None) -> str:
        """
        Send the stopword list to Claude and return the raw response string.
 
        Parameters
        ----------
        report  : output of StopwordsInvestigator.basic_report()
        samples : output of get_sample_reviews(), or None
        """
        issues_block = self._build_issues_block(report)
        real_data_block = self._build_real_data_block(samples)
        prompt = self._build_prompt(report, issues_block, real_data_block)
 
        print("[Claude] Sending audit request …")
        response = self.client.messages.create(
            model    = self.model,
            max_tokens = self.max_tokens,
            messages = [{"role": "user", "content": prompt}],
        )
        raw = response.content[0].text
 
        # Debug: show raw SUGGESTED_ADDITIONS block so format issues are visible
        m = re.search(r"SUGGESTED_ADDITIONS:\s*(.*?)(?=\n[A-Z_]+:|\Z)", raw, re.DOTALL)
        if m:
            print(f"\n[Debug] Raw SUGGESTED_ADDITIONS block:\n{m.group(1)[:600]}")
        else:
            print("\n[Debug] SUGGESTED_ADDITIONS section not found in response.")
 
        return raw
 
    # ── Response parsing ──────────────────────────────────────────────────────
 
    @staticmethod
    def parse_response(text: str) -> dict:
        """
        Extract AUDIT_ISSUES, SUGGESTED_ADDITIONS, and SUMMARY from Claude's reply.
 
        Returns
        -------
        dict with keys:
          audit_issues : list[dict]  — [{word, reason}, …]
          suggestions  : list[str]   — clean, validated words
          summary      : str
        """
        def extract_section(label: str) -> str:
            m = re.search(rf"{label}:\s*(.*?)(?=\n[A-Z_]+:|\Z)", text, re.DOTALL)
            return m.group(1).strip() if m else ""
 
        audit_raw   = extract_section("AUDIT_ISSUES")
        suggest_raw = extract_section("SUGGESTED_ADDITIONS")
        summary_raw = extract_section("SUMMARY")
 
        # ── Parse audit issues: "<word> | <reason>" ──────────────────────────
        audit_issues: list[dict] = []
        for line in audit_raw.splitlines():
            line = line.strip()
            if "|" in line:
                word, reason = line.split("|", 1)
                audit_issues.append({
                    "word"  : word.strip().lower(),
                    "reason": reason.strip(),
                })
 
        # ── Parse suggestions — robust multi-format cleaning ─────────────────
        suggestions:      list[str]         = []
        suggestions_skipped: list[tuple]    = []
 
        for line in suggest_raw.splitlines():
            # Strip leading: numbers, dots, dashes, bullets, em-dashes, whitespace
            word = re.sub(r"^[\s\d.\-*•–—]+", "", line).strip().lower()
            # Strip trailing punctuation
            word = word.rstrip(".,;:!? ")
 
            if not word:
                continue
            if len(word) < 2:
                suggestions_skipped.append((word, "len<2"))
                continue
            if not TOKENIZER_PATTERN.fullmatch(word):
                suggestions_skipped.append((word, f"regex mismatch"))
                continue
            suggestions.append(word)
 
        if suggestions_skipped:
            print(f"[Parser]  {len(suggestions_skipped)} words skipped:")
            for w, reason in suggestions_skipped:
                print(f"            '{w}' → {reason}")
        print(f"[Parser]  {len(suggestions)} suggestions accepted.")
 
        return {
            "audit_issues": audit_issues,
            "suggestions" : suggestions,
            "summary"     : summary_raw,
        }
 
    # ── Build enhanced list ───────────────────────────────────────────────────
 
    @staticmethod
    def build_enhanced_list(report: dict, parsed: dict) -> list[str]:
        """
        Produce the final enhanced stopword list:
          • Start from the original unique words
          • Drop single-char and regex-invalid words (they can't appear post-tokenization)
          • Remove words Claude flagged as non-stopwords
          • Add Claude's validated suggestions
 
        Parameters
        ----------
        report : output of StopwordsInvestigator.basic_report()
        parsed : output of parse_response()
        """
        flagged = {item["word"] for item in parsed["audit_issues"]}
        base: set[str] = set()
 
        for w in report["unique_list"]:
            if len(w) < 2:
                continue                          # Step 4 drops these anyway
            if not TOKENIZER_PATTERN.fullmatch(w):
                continue                          # regex can never capture these
            if w in flagged:
                continue                          # Claude says: keep in vocabulary
            base.add(w)
 
        # Track what actually changes
        actually_added:  list[str] = []
        already_present: list[str] = []
 
        for w in parsed["suggestions"]:
            if w in base:
                already_present.append(w)
            else:
                base.add(w)
                actually_added.append(w)
 
        # Logging
        if already_present:
            print(f"[Builder] {len(already_present)} suggestions already present: {already_present}")
        if actually_added:
            print(f"[Builder] {len(actually_added)} genuinely new words added:  {actually_added}")
        else:
            print("[Builder] No new words added — all suggestions already existed in the list.")
 
        return sorted(base)
 
    # ── Private prompt builders ───────────────────────────────────────────────
 
    @staticmethod
    def _build_issues_block(report: dict) -> str:
        issues: list[str] = []
        if report["duplicates"]:
            issues.append(f"- Duplicates found: {report['duplicates']}")
        if report.get("invalid_tokens"):
            issues.append(
                f"- Words that won't survive the tokenizer regex: {report['invalid_tokens']}"
            )
        if report["single_char"]:
            issues.append(
                f"- Single-char words (Step 4 removes len<2): {report['single_char']}"
            )
        return "\n".join(issues) if issues else "None detected."
 
    @staticmethod
    def _build_real_data_block(samples: dict | None) -> str:
        if not samples:
            return textwrap.dedent("""
                ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                REAL DATA — not available
                ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                No post-Step-4 data was provided. Base your audit on domain knowledge alone.
            """).strip()
 
        top_token_lines = "\n".join(
            f"  {rank+1:>3}. {tok:<25} (freq: {cnt})"
            for rank, (tok, cnt) in enumerate(samples["top_tokens"])
        )
        stream_lines = "\n".join(
            f"  [{i+1:>3}] {stream}"
            for i, stream in enumerate(samples["sampled_streams"][:20])
        )
        return textwrap.dedent(f"""
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            REAL DATA — REVIEWS AFTER STEPS 1-4 (before stopword removal)
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            Source  : {samples['n_sampled']} randomly sampled reviews out of {samples['total_reviews']:,} total.
            Pipeline: tokenized (regex) → lowercased → len<2 removed.
            Stopwords have NOT been applied yet — this is what the filter sees.
 
            TOP {len(samples['top_tokens'])} MOST FREQUENT TOKENS IN THE SAMPLE
            (Pure filler words appearing here are strong candidates to add as stopwords.)
            {top_token_lines}
 
            SAMPLE TOKEN STREAMS (20 of {samples['n_sampled']} reviews shown)
            {stream_lines}
        """).strip()
 
    @staticmethod
    def _build_prompt(report: dict, issues_block: str, real_data_block: str) -> str:
        return textwrap.dedent(f"""
            You are an NLP preprocessing expert reviewing a stopword list that sits
            inside a multi-stage text analytics pipeline for cosmetics and beauty product reviews
            (~161,200 reviews). Understanding the full pipeline is critical for judging which
            words should and should not be removed.
 
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            PIPELINE OVERVIEW & PURPOSE OF EACH STAGE
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 
            TASK 1 — Text Pre-processing (this is where stopword removal lives)
              Goal : Convert raw review text into a clean, consistent token stream and build
                     a shared vocabulary that all downstream tasks depend on.
              Steps (in order):
                1. Extract review_text column only
                2. Tokenize with regex r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"  ← captures contractions
                3. Lowercase all tokens
                4. Remove tokens with length < 2
                5. Remove stopwords  ← THIS LIST IS WHAT YOU ARE AUDITING
                6. Remove words with term frequency = 1 across the whole corpus
                7. Remove the top-20 most frequent words by document frequency
                8. Save processed.csv
                9. Build vocab.txt (word:index, alphabetically sorted, index from 0)
 
            TASK 2 — Feature Representation  (consumes vocab.txt + processed.csv)
              Three parallel branches, all using review_text only:
                A. Bag-of-Words — sparse word:freq encoding based on vocab.txt
                   → Stopwords inflate the most-frequent columns with zero signal.
                B. Unweighted embeddings — plain average of pretrained word vectors
                   → Filler words drag the document centroid toward a generic centre.
                C. TF-IDF weighted embeddings — vectors weighted before averaging
                   → Stopwords have near-zero IDF, but removing them avoids edge cases.
 
            TASK 3 — Classification  (target: is_a_buyer True/False, ~161k reviews)
              Q1: Compare BoW vs unweighted vs TF-IDF-weighted embeddings.
              Q2: Does adding review_title, brand, price, avg_rating improve accuracy?
              Classifiers: Logistic Regression, LinearSVC, RandomForest.
              Primary metric: F1-score (classes may be imbalanced).
 
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            WHAT A GOOD STOPWORD MEANS IN THIS CONTEXT
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            REMOVE if the word:
              • Carries no signal about whether a reviewer bought the product
              • Contributes nothing to the semantic meaning of a beauty review
              • Would waste a vocabulary slot or pull embeddings toward a generic centre
              Examples: function words, generic discourse markers, informal filler
 
            KEEP (do NOT mark as stopword) if the word:
              • Could correlate with purchase intent (sentiment, action, evaluation words)
              • Carries domain-relevant meaning in cosmetics reviews
              • Would genuinely help distinguish buyers from non-buyers in Task 3
 
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            CURRENT STOPWORD LIST ({report['unique_words']} unique words)
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            {', '.join(report['unique_list'])}
 
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            PRE-FLIGHT ISSUES DETECTED BY LOCAL CHECKS
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            {issues_block}
 
            {real_data_block}
 
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            YOUR TASK
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            1. AUDIT  — Words currently in the list that should NOT be stopwords.
                        One entry per line: <word> | <reason tied to Task 2 or Task 3 impact>
            2. GAPS   — Up to 30 words MISSING from the list that are safe to add.
                        Rules: match regex r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?", length ≥ 2,
                        pure noise with zero purchase-intent signal.
                        One word per line. Do NOT include words already in the list above.
            3. SUMMARY — One paragraph on the overall list quality for this pipeline.
 
            Reply in EXACTLY this format — no extra headers, no preamble:
 
            AUDIT_ISSUES:
            <word> | <reason>
 
            SUGGESTED_ADDITIONS:
            <word>
 
            SUMMARY:
            <paragraph>
        """).strip()

### 1.1 Examining and loading data
- Examine the data and explain your findings
- Load the data into proper data structures and get it ready for processing.

In [54]:
# Code to inspect the provided data file...
df_raw: pd.DataFrame = pd.read_csv("./data/cosmetics_beauty_products_reviews.csv")

In [55]:
# Quick view
df_raw.shape

(61284, 15)

In [56]:
df_raw.head()

,product_id,brand_name,review_id,review_title,review_text,author,review_date,review_rating,is_a_buyer,product_title,price,avg_product_rating,product_rating_count,product_tags,product_url
0,781070,Olay,16752142,Worth buying 50g one,Works as it claims. Could see the difference f...,Ashton Dsouza,23/01/2021 15:17,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
1,781070,Olay,14682550,Best cream to start ur day,It does what it claims . Best thing is it smoo...,Amrit Neelam,07/09/2020 15:30,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
2,781070,Olay,15618995,perfect for summers dry for winters,I have been using this product for months now....,Sanchi Gupta,13/11/2020 12:24,4.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
3,781070,Olay,13474509,Not a moisturizer,"i have an oily skin, while this whip acts as a...",Ruchi Shah,14/06/2020 11:56,3.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
4,781070,Olay,16338982,Average,It's not that good. Please refresh try for oth...,Sukanya Sarkar,22/12/2020 15:24,2.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...


In [57]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61284 entries, 0 to 61283
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_id            61284 non-null  int64  
 1   brand_name            61284 non-null  object 
 2   review_id             61284 non-null  int64  
 3   review_title          61284 non-null  object 
 4   review_text           61275 non-null  object 
 5   author                61284 non-null  object 
 6   review_date           61284 non-null  object 
 7   review_rating         61283 non-null  float64
 8   is_a_buyer            61284 non-null  bool   
 9   product_title         61284 non-null  object 
 10  price                 61284 non-null  int64  
 11  avg_product_rating    61284 non-null  float64
 12  product_rating_count  61284 non-null  int64  
 13  product_tags          13502 non-null  object 
 14  product_url           61284 non-null  object 
dtypes: bool(1), float64

### 1.2 Pre-processing data
Perform the required text pre-processing steps.

...... Sections and code blocks on basic text pre-processing


<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

### Plans
<img src="https://raw.githubusercontent.com/DatacollectorVN/RMIT-Advanced-Programming-for-Data-Science/58743db2d9da49cf8de6f4eaef5c5fda06034c82/assigments/group/publics/task1_nlp_pipeline_stages.svg" style="width:60%;">

####  Stop Words Enhancement
In Step4 we want to apply the AI agent to review the stopwords_en.txt that statisfy for that assignment's purpose or not.

*Note*: Enhancing Stopword (Please skip that code below if you dont want use Claude to audit)

In [ ]:
# Audit sample
samples = ClaudeTextExpert.get_sample_reviews(df_raw) # Get sample reviews
original_words = load_stopwords(STOPWORDS_PATH) # Load original stopwords
report = basic_report(original_words) # Basic report
print(f"[Loaded] {report['total_lines']} lines, {report['unique_words']} unique words")


[Sampler] 61,284 reviews received — sampled 300 for Claude context.
[Loaded] 571 lines, 570 unique words


In [ ]:
expert: ClaudeTextExpert = ClaudeTextExpert() # ClaudeTextExpert
raw_response: str = expert.claude_audit(report, samples) # Claude audit


[Claude] Sending audit request …

[Debug] Raw SUGGESTED_ADDITIONS block:
ok
oh
omg
ha
hey
hm
ah
yep
nope
gonna
gotta
wanna
kinda
sorta
lol
hmm
ugh
tbh
tbf
imo
idk
etc
eg
ie
vs
nd
rd
th
sup
uucp



In [ ]:
# 5. Parse
parsed = parse_claude_response(raw_response)

# 6. Build enhanced list
enhanced = build_enhanced_list(original_words, parsed, report)

# 7. Decide whether to save
original_clean = sorted(set(
    w for w in report["unique_list"]
    if len(w) >= 2 and TOKENIZER_PATTERN.fullmatch(w)
))
needs_update = enhanced != original_clean


In [67]:
if needs_update:
    save_enhanced(enhanced, ENHANCED_PATH)
else:
    print("\n[Info] No changes needed — enhanced list is identical to cleaned original.")
    print(f" {ENHANCED_PATH} was NOT created.")



[Saved] 505 stopwords → ./data/stopwords_en_enhancement.txt


In [69]:
print_report(report, parsed, enhanced, report["unique_list"])


  STOPWORDS INVESTIGATION REPORT

[Original file]
  Total lines  : 571
  Unique words : 570
  Duplicates   : ['would']
  Single-char (Step 4 removes): ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

[Claude Audit — words to reconsider]
  - best                 Strong evaluative signal; "best" correlates with high satisfaction and purchase intent — directly useful for Task 3 buyer classification
  - better               Comparative sentiment word; distinguishes satisfied from dissatisfied reviewers, valuable for Task 3 F1
  - good                 High-frequency positive sentiment marker (rank 13 in sample); carries clear purchase-intent signal for buyer classification
  - like                 Can express preference/similarity to desired outcome ("I like", "looks like silk"); retains sentiment signal relevant to Task 3
  - love                 Strong positive sentiment and purchase-intent signal (rank 30

In [85]:
def preprocess(texts: list[str], stopwords: list[str], top_k_df: int = 20) -> list[list[str]]:
    """
    Apply Task1 Step 2 -> Step 7 on a collection of documents.
    Returns: list of token lists (one per document).
    """
    stop_set = set(w.lower().strip() for w in stopwords if w.strip())
    # Step 2-5: tokenize, lowercase, len>=2, remove stopwords
    docs = []
    for text in texts:
        tokens = TOKENIZER_PATTERN.findall(str(text))          # Step 2
        tokens = [t.lower() for t in tokens]                   # Step 3
        tokens = [t for t in tokens if len(t) >= 2]            # Step 4
        tokens = [t for t in tokens if t not in stop_set]      # Step 5
        docs.append(tokens)
    
    # Step 6: remove words with term frequency == 1 in whole collection
    tf = Counter(t for doc in docs for t in doc)
    docs = [[t for t in doc if tf[t] > 1] for doc in docs]
    
    # Step 7: remove top-20 most frequent words by document frequency
    df = Counter()
    for doc in docs:
        df.update(set(doc))  # each word counted once per document
    top_df_words = {w for w, _ in df.most_common(top_k_df)}
    docs = [[t for t in doc if t not in top_df_words] for doc in docs]
    return docs


def build_vocab(processed_docs: list[list[str]], path: str) -> dict:
    """
    Build a vocabulary from a list of tokenized documents.
    Returns: dict with word:index pairs, sorted alphabetically.
    """
    vocab_set = {t for doc in processed_docs for t in doc}
    vocab_sorted = sorted(vocab_set) # # A–Z (ASCII-ish; digits would sort before letters if any)
    lines = []
    for i, word in enumerate(vocab_sorted):
        lines.append(f"{word}:{i}")
    
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print(f"\n[Saved] {len(lines)} words → {path}")

In [78]:
stopwords = load_stopwords(ENHANCED_PATH)  # or STOPWORDS_PATH
processed_docs = preprocess(df_raw["review_text"].fillna("").tolist(), stopwords)

In [79]:
processed_docs[:5]

[['works', 'claims', 'difference', 'day', 'olay', 'cleanser', 'results'],
 ['claims', 'thing', 'smoothens', 'ur', 'makes', 'soft'],
 ['using',
  'months',
  'combination',
  'oily',
  'greasy',
  'absorbs',
  'quickly',
  'moisturises',
  'doesnt',
  'work',
  'winters'],
 ['oily',
  'whip',
  'acts',
  'great',
  'base',
  'primer',
  'smoothens',
  'moisturise',
  'felt',
  'needed',
  'moisturiser',
  'worth',
  'price',
  'buying'],
 ['please', 'refresh', 'try', 'products']]

['works claims difference day olay cleanser results',
 'claims thing smoothens ur makes soft',
 'using months combination oily greasy absorbs quickly moisturises doesnt work winters',
 'oily whip acts great base primer smoothens moisturise felt needed moisturiser worth price buying',
 'please refresh try products',
 'dz dry olay representative suggest buy dz type please oily type don buys dry normal packaging box big size bt inside ll gt lipbalm size cream great loss small quantity heavy price',
 'cream awesome makes rough soft leaving oily effect continue',
 'instantly tone appearance',
 'eye cream combo effective works fine lines eye dark circles worth penny',
 'helps reduces dark spots wrinkles',
 'combo olay absolutely worth',
 'awesome',
 'increases pimples',
 'scented cream give honest review weeks olay disappoint',
 'suits thing trust brand',
 'superb purchase effective',
 'olay works wonders combo feels soft hydrated compliments follow diet cut sugar oils food blessed',
 'time 

## Saving required outputs
Save the requested information as per specification.
- vocab.txt

In [90]:
# code to save output data...
build_vocab(processed_docs, VOCAB_PATH)
df_final = df_raw.copy()
df_final["processed_review_text"] = [" ".join(tokens) for tokens in processed_docs]
df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")


[Saved] 8095 words → ./data/vocab.txt


## Summary
Give a short summary and anything you would like to talk about the assessment task here.

We first audited the provided stopword list with the Claude API: loaded stopwords_en.txt, generated a quality report (duplicates/invalid/single-char), sampled real review tokens (after basic tokenization/cleanup), asked Claude which words to remove/keep/add for this assignment context, parsed the response, and saved an enhanced list as stopwords_en_enhancement.txt.

Then we ran the Task 1 preprocessing pipeline on review_text: Step 2 regex tokenization, Step 3 lowercase, Step 4 remove tokens with length < 2, Step 5 remove stopwords (using chosen stopword file), Step 6 remove corpus terms with term frequency = 1, and Step 7 remove top-20 words by document frequency. Finally, we produced Step 8 processed.csv (with processed_review_text) and Step 9 vocab.txt (unique processed tokens sorted A→Z with index from 0).

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>